# Assignment 5: Data Cleaning and Data Model Building using Python

## AIM
To perform different data cleaning and data model building operations 
using Python on Air Quality and Heart Disease datasets.

## OBJECTIVES
- To learn data processing methods
- To learn building predictive models
- To understand preprocessing techniques used in real-world analytics

## Tools Used
- Python
- Pandas
- NumPy
- Scikit-learn
- VS Code (Jupyter Notebook)

In [13]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

## Loading the Datasets

Air Quality dataset contains pollution parameters.
Heart Disease dataset contains patient health attributes.

These datasets contain:
- Missing values
- Outliers
- Different data types

In [14]:
air_df = pd.read_csv("air_quality.csv")
heart_df = pd.read_csv("heart_disease.csv")

air_df.head()
heart_df.head()

,PatientID,Age,Cholesterol,BloodPressure,HeartDisease
0,1,45,230.0,130.0,1
1,2,50,250.0,140.0,1
2,3,38,NaN,120.0,0
3,4,60,300.0,160.0,1
4,5,35,180.0,110.0,0


### Checking Missing Values

Missing values are represented as NaN.
We must detect and handle them before analysis.

In [15]:
air_df.isnull().sum()
heart_df.isnull().sum()

PatientID        0
Age              0
Cholesterol      1
BloodPressure    1
HeartDisease     0
dtype: int64

### Missing Value Treatment

Approach Used:
- Numerical columns → Replace with Mean or Median

Why Median?
Median is robust to outliers.

### Important Note on Chained Assignment

Using `inplace=True` on a sliced column may raise a FutureWarning in Pandas.

Instead of:
df['column'].fillna(value, inplace=True)

We use:
df['column'] = df['column'].fillna(value)

This ensures modification happens on the original DataFrame 
and avoids chained assignment issues.

In [16]:
# Air Quality
air_df['NO2'] = air_df['NO2'].fillna(air_df['NO2'].mean())
air_df['Temperature'] = air_df['Temperature'].fillna(air_df['Temperature'].mean())

# Heart Disease
heart_df['Cholesterol'] = heart_df['Cholesterol'].fillna(heart_df['Cholesterol'].median())
heart_df['BloodPressure'] = heart_df['BloodPressure'].fillna(heart_df['BloodPressure'].mean())

heart_df

,PatientID,Age,Cholesterol,BloodPressure,HeartDisease
0,1,45,230.0,130.000000,1
1,2,50,250.0,140.000000,1
2,3,38,230.0,120.000000,0
3,4,60,300.0,160.000000,1
4,5,35,180.0,110.000000,0
5,6,70,500.0,180.000000,1
6,7,55,220.0,137.857143,1
7,8,40,190.0,125.000000,0


### Outlier Detection using IQR Method

IQR = Q3 - Q1

Values outside:
Q1 - 1.5 * IQR
Q3 + 1.5 * IQR

are treated as outliers.

In [17]:
Q1 = heart_df['Cholesterol'].quantile(0.25)
Q3 = heart_df['Cholesterol'].quantile(0.75)
IQR = Q3 - Q1

lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR

heart_df['Cholesterol'] = np.where(
    heart_df['Cholesterol'] > upper,
    upper,
    heart_df['Cholesterol']
)

heart_df

,PatientID,Age,Cholesterol,BloodPressure,HeartDisease
0,1,45,230.0,130.000000,1
1,2,50,250.0,140.000000,1
2,3,38,230.0,120.000000,0
3,4,60,300.0,160.000000,1
4,5,35,180.0,110.000000,0
5,6,70,337.5,180.000000,1
6,7,55,220.0,137.857143,1
7,8,40,190.0,125.000000,0


## Data Integration

Data Integration means combining data from multiple sources.

Here we demonstrate integration using concat().

In [18]:
combined_df = pd.concat([air_df, heart_df], axis=0, ignore_index=True)
combined_df.head()

,City,PM2.5,NO2,SO2,Temperature,PatientID,Age,Cholesterol,BloodPressure,HeartDisease
0,Mumbai,120.0,45.0,30.0,32.0,NaN,NaN,NaN,NaN,NaN
1,Delhi,200.0,44.0,50.0,35.0,NaN,NaN,NaN,NaN,NaN
2,Pune,90.0,30.0,20.0,28.0,NaN,NaN,NaN,NaN,NaN
3,Chennai,150.0,40.0,35.0,31.6,NaN,NaN,NaN,NaN,NaN
4,Kolkata,300.0,80.0,60.0,34.0,NaN,NaN,NaN,NaN,NaN


### Feature Scaling (Standardization)

Scaling ensures:
- Mean = 0
- Standard Deviation = 1

This is required before model building 
to ensure all variables are treated equally.

Log transformation should be applied before feature scaling.

Applying log after standardization may lead to invalid values 
because scaling produces negative numbers, and logarithm of 
negative numbers is undefined.

In [19]:
# Step 1: Log transform BEFORE scaling
heart_df['Cholesterol_log'] = np.log(heart_df['Cholesterol'])

# Step 2: Then scale
scaler = StandardScaler()
heart_df[['Age','Cholesterol_log','BloodPressure']] = scaler.fit_transform(
    heart_df[['Age','Cholesterol_log','BloodPressure']]
)

heart_df.head()

,PatientID,Age,Cholesterol,BloodPressure,HeartDisease,Cholesterol_log
0,1,-0.366599,230.0,-0.370757,1,-0.158648
1,2,0.077763,250.0,0.101116,1,0.261396
2,3,-0.988706,230.0,-0.842630,0,-0.158648
3,4,0.966488,300.0,1.044861,1,1.179858
4,5,-1.255323,180.0,-1.314503,0,-1.393476


### Resolving Skewness

If data is heavily right-skewed,
log transformation helps normalize distribution.

## Error Correction

Data type correction and validation are important.

Example:
Convert Age to integer.

In [20]:
heart_df['Age'] = heart_df['Age'].astype(int)
heart_df.dtypes

PatientID            int64
Age                  int64
Cholesterol        float64
BloodPressure      float64
HeartDisease         int64
Cholesterol_log    float64
dtype: object

### Preparing Data for Model

Predictor Variables (X):
- Age
- Cholesterol
- BloodPressure

Target Variable (y):
- HeartDisease

In [21]:
X = heart_df[['Age','Cholesterol','BloodPressure']]
y = heart_df['HeartDisease']

### Train-Test Split

We divide data into:
- Training data (70%)
- Testing data (30%)

In [22]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)

### Logistic Regression Model

Used for binary classification.
HeartDisease is 0 or 1 → Binary Outcome.

In [23]:
model = LogisticRegression()
model.fit(X_train, y_train)

predictions = model.predict(X_test)

accuracy = accuracy_score(y_test, predictions)
accuracy

1.0

## Conclusion

In this assignment, we performed:

- Data Cleaning
- Missing Value Treatment
- Outlier Detection and Treatment
- Data Integration
- Data Transformation
- Error Correction
- Model Building using Logistic Regression

We understood how raw data is converted into consistent and 
analysis-ready data.

These preprocessing steps are essential in real-world 
machine learning pipelines.